<div style="background-color:#B5121B;color:#ffffff;padding:16px 22px;border-radius:6px;font-family:Arial, Helvetica, sans-serif;">
<div style="font-size:12px;letter-spacing:0.08em;text-transform:uppercase;opacity:0.85;">PHYS7101 &middot; Numerical Methods and Data Analysis</div>
<div style="font-size:21px;font-weight:700;margin-top:6px;line-height:1.3;">Week 8 — Learning from labelled data</div>
</div>

#### Classifying stars, galaxies and quasars and predicting galaxy redshifts from SDSS colours: trees, forests and nearest neighbours, cross-validation, precision and recall on the real sky, and the limits of extrapolation.
***

## The question

In Week 6 you found structure in the SDSS colours *without* labels. Here each object also has a spectrum, which tells us what it really is (star, galaxy or quasar) and how far away it is (its redshift). Spectra are expensive: a few million objects have one, while imaging gives colours for over a billion. So: **from the cheap measurements (colours and size), how well can we predict the expensive ones (class and redshift), and how do we know?**

That last part is the point of today. Lecture 10 introduced decision trees, nearest neighbours and cross-validation; the toolkit adds random forests, precision and recall, and regression. Here you learn to judge them honestly, which is exactly what the data-driven half of Exercise 3 asks for.

1. **Classify:** a tree, nearest neighbours and a forest, with cross-validation to choose their settings (about 25 minutes).
2. **Judge it on the real sky:** quasars are rare; accuracy hides that (about 25 minutes).
3. **Predict a number:** galaxy redshift from colours, and what happens beyond the training range (about 25 minutes).
4. *Optional:* which features matter.

**Exercise 2 is due at 4pm on Thursday:** if you would rather spend the lab on it, do; these notebooks will keep. The [Week 8 toolkit](week8_python_toolkit.ipynb) has every scikit-learn tool used here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, KFold, GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score
from sklearn.inspection import permutation_importance

In [ ]:
import os

# Data: read from a local datasets/ folder if you've cloned/downloaded the
# repository, otherwise straight from GitHub (e.g. in Colab). Same file either way.
DATA_URL = "https://raw.githubusercontent.com/MatSmithAstro/phys7101_2026_resources/main/workshops/week8/datasets/"

def data_path(filename):
    local = os.path.join("datasets", filename)
    return local if os.path.exists(local) else DATA_URL + filename

In [ ]:
def features(d):
    """Colours (differences of magnitudes) and the log of the size, as in Week 6."""
    return pd.DataFrame({"u_g": d.mag_u - d.mag_g, "g_r": d.mag_g - d.mag_r, "r_i": d.mag_r - d.mag_i,
                         "i_z": d.mag_i - d.mag_z, "log_size": np.log10(d.size_r)})

bal = pd.read_csv(data_path("sdss_balanced.csv"))      # 600 of each class
sky = pd.read_csv(data_path("sdss_sky.csv"))           # 10 947 objects in their natural proportions
print(bal["class"].value_counts().to_dict(), sky["class"].value_counts().to_dict())

## <span style="color:#B5121B">1. Classify</span>

Start with the balanced set, where each class is a third of the data. Hold back 30% as a test set and do not touch it until the end of this section.

In [ ]:
X, y = features(bal), bal["class"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=0)

**1.1 A tree you can read.** Fit a `DecisionTreeClassifier(max_depth=3, random_state=0)` to the training data, print its rules with `print(export_text(tree, feature_names=list(X.columns)))`, and find its training and test accuracy. Which feature does it split on first, and does that make physical sense? (Think about what a star looks like on an image.)

In [ ]:
# 1.1

**1.2 How deep?** Cross-validate trees of depth 1, 2, 3, 5, 8 and no limit (5 folds, training data only), and print the training accuracy alongside. Then do the same for k-nearest neighbours with k = 1, 5, 15, 30 and 60, inside a pipeline with `StandardScaler`. Which settings would you choose?

In [ ]:
# 1.2
for d in (1, 2, 3, 5, 8, None):
    ...
for k in (1, 5, 15, 30, 60):
    ...

**Check:** trees: depth 1 scores 0.660 (one split can only separate one class); depths 2 to 8 all give 0.945–0.950 ± 0.015, while the training accuracy climbs from 0.950 to 1.000. k-NN: 0.963, 0.964, 0.954, 0.944, 0.928 for k = 1, 5, 15, 30, 60.

**1.3 The verdict.** Fit your chosen tree, your chosen k-NN and a `RandomForestClassifier(300, random_state=0, n_jobs=-1)` to the whole training set, and find each one's accuracy on the test set. Is the forest's advantage bigger than the uncertainty you saw in 1.2?

In [ ]:
# 1.3

**Check:** tree 0.950, k-NN (k = 5) 0.965, forest 0.969, on 540 test objects.

## <span style="color:#B5121B">2. Judge it on the real sky</span>

On the real sky only 6% of objects are quasars. Split `sky` 70/30 (stratified), then compare three ways of training a `RandomForestClassifier(300, random_state=0, n_jobs=-1)`, all tested on the same natural test set:

- **balanced:** train on a subsample of the training set with equal numbers of each class (as many as there are quasars);
- **natural:** train on the whole training set;
- **natural, weighted:** the whole training set with `class_weight="balanced"`.

For each, print the accuracy and the `classification_report`. Also work out the accuracy of a "model" that always says *galaxy*.

In [ ]:
Xs, ys = features(sky), sky["class"]
S_train, S_test = train_test_split(sky, test_size=0.3, stratify=ys, random_state=0)
n_q = (S_train["class"] == "qso").sum()
S_bal = S_train.groupby("class").sample(n_q, random_state=0)     # n_q of each class
print("quasars in the training set:", n_q, "  test set:", (S_test["class"] == "qso").sum(), "of", len(S_test))

In [ ]:
# 2.1

**Check:** balanced training: accuracy 0.978, quasar precision 0.804, recall 0.928. Natural: 0.988, 0.961, 0.892. Natural, weighted: 0.987, 0.946, 0.897. Always *galaxy*: 0.576. There are 194 quasars in the test set.

**2.2 Questions.** (a) Why does training on the balanced set give *lower* quasar precision on the real sky, even though it finds more of them? (b) An astronomer wants a list of quasar candidates to observe with an expensive spectrograph. Which version would you give them, and which number would you quote? (c) Is 0.98 accuracy impressive here?

**2.3 Holding out regions of sky.** A random split puts neighbouring objects in both training and test sets. If the data differ from one part of the sky to another (dust, observing conditions), that flatters the model. Use `GroupKFold(5)` with groups `sky["ra"] // 30` (twelve strips of sky, 30° wide in right ascension) and compare the cross-validated accuracy and quasar precision and recall with a random `KFold(5, shuffle=True, random_state=0)`. `cross_val_predict` gives every object a prediction from the fold in which it was held out.

In [ ]:
region = sky["ra"] // 30
def qso_scores(true, pred):
    return precision_score(true, pred, labels=["qso"], average=None)[0], recall_score(true, pred, labels=["qso"], average=None)[0]

In [ ]:
# 2.3

**Check:** both splits give accuracy 0.987–0.988, quasar precision 0.94 and recall 0.904.

## <span style="color:#B5121B">3. Predict a number: galaxy redshift</span>

Now regression: predict each galaxy's redshift *z* from its colours and its r magnitude (fainter galaxies tend to be further away). This is a "photometric redshift", how most galaxies in modern surveys get a distance.

In [ ]:
gal = sky[sky["class"] == "galaxy"].copy()
Xg = features(gal).drop(columns="log_size").assign(mag_r=gal.mag_r)
zg = gal["redshift"]
print(len(gal), "galaxies; redshift median", round(zg.median(), 3), " 5–95%:", zg.quantile([0.05, 0.95]).round(3).tolist())
G_train, G_test, z_train, z_test = train_test_split(Xg, zg, test_size=0.3, random_state=0)

**3.1** Fit a `LinearRegression`, a `KNeighborsRegressor(15)` (in a pipeline with `StandardScaler`) and a `RandomForestRegressor(300, min_samples_leaf=5, random_state=0, n_jobs=-1)`. For each, find the RMSE on the test set. Plot predicted against true redshift for the forest, with the line *predicted = true*. How does the RMSE compare with the spread of the redshifts themselves?

In [ ]:
# 3.1

**Check:** RMSE 0.027 (line), 0.024 (k-NN), 0.023 (forest), against a spread of 0.061 in the redshifts themselves: the colours carry most of the information. The points follow the line, with the highest redshifts tending to be predicted too low.

**3.2 Beyond the training range.** Suppose the spectra only reached nearby galaxies. Train each model on galaxies with *z* < 0.12 and test on those with *z* > 0.18. Print the mean predicted redshift and the RMSE for each, and compare with the true mean.

In [ ]:
# 3.2

**Check:** 3518 training and 969 test galaxies, true mean 0.229. Straight line: mean prediction 0.156, RMSE 0.079. k-NN and forest: about 0.105, RMSE about 0.13.

**3.3 Question.** Why do the forest and k-NN stop at about the same value? Why does the straight line do better, and would you trust it further still? What does this suggest about when a physical model earns its place over a data-driven one, which is the comparison Exercise 3 asks you to make?

## <span style="color:#FD9029">4. Optional: which features matter?</span>

Use `permutation_importance` (20 repeats, on the test set) for the forest classifier of Section 1 and the forest regressor of Section 3. Which feature does each depend on most? Does that match the first split of the tree in 1.1?

In [ ]:
# 4

**Check:** class: `u_g` and `log_size` matter about equally (accuracy drops by about 0.33 when either is shuffled), then `i_z`. Redshift: `g_r` dominates (the regressor's score is R², which falls by about 1.8 when `g_r` is shuffled), then `u_g`; the r magnitude adds little.

## <span style="color:#38614A">Now: Exercise 3</span>

Exercise 2 first: it is due at 4pm on Thursday. Once it is in, Exercise 3 (on Moodle) is next: read the brief and the "How your work is marked" page, choose project A or B, and run its starter notebook. A good first target: your data loaded, one plot, and a sentence saying what your physical model will be. What you did here carries straight over:

- **choose settings by cross-validation**, and keep a test set for the end;
- **hold out whole groups** (stations, earthquake sequences) if your question is about ones you have not seen;
- **quote the number that matters** for the question, with its uncertainty, against a simple baseline;
- **check whether you are asking the model to extrapolate**: if so, that is where the physical model may win.

<hr style="border:none;border-top:1px solid #BEC0C2;margin-top:32px;">
<div style="color:#555656;font-size:12px;margin-top:8px;font-family:Arial, Helvetica, sans-serif;">
PHYS7101 &mdash; Numerical Methods and Data Analysis
</div>